# 495 – Ohsome analysis of SAT objects

Retrieve and analyse every OpenStreetMap object carrying `ref:stockholmarchipelagotrail`.

- [Issue #495](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/495)
- [Ohsome API documentation](https://docs.ohsome.org/ohsome-api/v1/)
- [OSM key documentation](https://wiki.openstreetmap.org/wiki/Sv:Key:ref:stockholmarchipelagotrail)

The filter is deliberately not restricted to ways: nodes, ways and relations are all included.


In [ ]:
from datetime import date

import pandas as pd
import requests

OHSOME_API = "https://api.ohsome.org/v1"
FILTER = "ref:stockholmarchipelagotrail=*"
START = "2010-01-01"
END = date.today().isoformat()
TIME_SERIES = f"{START},{END},P1Y"

session = requests.Session()
session.headers.update({
    "User-Agent": "SAT-Ohsome-495/1.0 (https://github.com/salgo60/Stockholm_Archipelago_Trail)",
    "Accept": "application/json",
})


## Historical count

This is the reusable Ohsome query for the issue. Ohsome returns one feature per time interval, making changes in the number of tagged objects visible without downloading geometries.


In [ ]:
count_response = session.get(
    f"{OHSOME_API}/elements/count",
    params={"filter": FILTER, "time": TIME_SERIES},
    timeout=120,
)
count_response.raise_for_status()
count = count_response.json()
count


In [ ]:
count_rows = [
    {"timestamp": feature["timestamp"], "objects": feature["value"]}
    for feature in count["result"]
]
counts = pd.DataFrame(count_rows)
counts


## Current objects and tag analysis

The geometry endpoint downloads the current matching objects globally. The resulting GeoJSON can be saved or displayed on a map, while the table below provides an inventory by OSM element type and SAT reference value.


In [ ]:
objects_response = session.get(
    f"{OHSOME_API}/elements/geometry",
    params={"filter": FILTER, "time": END, "format": "geojson"},
    timeout=120,
)
objects_response.raise_for_status()
objects = objects_response.json()
features = objects.get("features", [])

rows = []
for feature in features:
    properties = feature.get("properties", {})
    rows.append({
        "osm_id": properties.get("@id"),
        "osm_type": properties.get("@osmType"),
        "sat_ref": properties.get("ref:stockholmarchipelagotrail"),
        "geometry_type": feature.get("geometry", {}).get("type"),
        "last_edit": properties.get("@lastEdit"),
    })

objects_table = pd.DataFrame(rows)
objects_table.head()


In [ ]:
summary = pd.DataFrame({
    "objects": [len(objects_table)],
    "distinct_sat_refs": [objects_table["sat_ref"].nunique(dropna=True)],
    "missing_geometry": [objects_table["geometry_type"].isna().sum()],
})
by_type = objects_table.groupby("osm_type", dropna=False).size().rename("objects")
by_ref = objects_table.groupby("sat_ref", dropna=False).size().rename("objects").sort_values(ascending=False)

display(summary)
display(by_type)
display(by_ref)


The raw response is retained in `objects`; it can be written to GeoJSON for further spatial analysis:

```python
import json
with open("495_ohsome_sat_objects.geojson", "w", encoding="utf-8") as output:
    json.dump(objects, output, ensure_ascii=False, indent=2)
```
